# HIGHFLY · OBSIDIAN MONARCH · 3D GRATUITO TODO AUTOMÁTICO (R0.6)

**Una sola operación:** Google Colab → GPU **T4 gratis** → **Entorno de ejecución > Ejecutar todo** → **subir imagen** cuando aparezca la ventana. Todo lo demás se instala/verifica/genera de manera automática, sin copiar código.

**R0.6 corrige el último fallo:** la importación de PartCrafter terminó con código 1, pero el cuaderno anterior ocultaba su error interno. Esta versión muestra los errores completos, utiliza versiones fijadas de Transformers/Diffusers acordes con el modelo original y elimina imports de render/OpenGL y RMBG no necesarios para producir GLB. Conserva el Python 3.11 aislado y evita apt-get.

**Costo:** cero créditos o suscripciones. La T4 gratuita de Colab no está garantizada en tiempo/disponibilidad. **Importante:** los cambios todavía deben probarse con inferencia real; un RUN GREEN de GitHub es una prueba CPU de sintaxis y seguridad, no prueba GPU/artística.

**Resultado esperado:** ZIP con object.glb, cuatro part_XX.glb y manifiesto. El GLB es **material geométrico sin rig ni ajuste al Warrior**; falta selección de piezas, Blender y prueba de 22 animaciones. No prometemos fidelidad exacta a la ilustración.


In [ ]:
# HIGHFLY SKIN FACTORY · R0.5 AUTO · NO EDITAR ESTA CELDA
import os, sys, subprocess, shutil, json, zipfile, hashlib, traceback, re, collections
from pathlib import Path
from io import BytesIO
from datetime import datetime, timezone

ROOT=Path("/content/PartCrafter")
VENV=Path("/content/highfly_partcrafter_py311")
PY=VENV/"bin/python"
CHECKPOINT="3d773bf02fad51c7ab31a5615573fec93b287b30"
TAG="highfly_obsidian_pilot"
OUTDIR=Path("/content/highfly_partcrafter_output")
TARGET=OUTDIR/TAG
ZIP=Path("/content/HIGHFLY_OBSIDIAN_PARTCRAFTER_RAW3D.zip")
CURRENT_STAGE="INICIO"

def stage(title):
    global CURRENT_STAGE
    CURRENT_STAGE=title
    print("\n"+"="*60+"\nHIGHFLY ► "+title+"\n"+"="*60,flush=True)

def run(cmd,cwd=None,timeout=1800,env=None):
    cmd=[str(x) for x in cmd]
    print(">", " ".join(cmd),flush=True)
    # Importantísimo: el Colab anterior ocultaba el traceback REAL del Python hijo.
    # Transmitir salida y stderr línea a línea, y conservar las últimas 100 líneas.
    recent=collections.deque(maxlen=100)
    with subprocess.Popen(cmd,cwd=cwd,env=env,stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT,bufsize=1,text=True,errors="replace") as proc:
        try:
            for line in proc.stdout:
                print(line,end="",flush=True)
                recent.append(line.rstrip())
            rc=proc.wait(timeout=timeout)
        except BaseException:
            proc.kill()
            proc.wait()
            raise
    if rc:
        print("\nHIGHFLY COMANDO FALLÓ, últimas líneas:",flush=True)
        print("\n".join(recent),flush=True)
        raise subprocess.CalledProcessError(rc,cmd)
    return True

def verify_partcrafter_patches():
    src=ROOT/"scripts/inference_partcrafter.py"
    text=src.read_text(encoding="utf-8")
    # Solo inferencia sin render ni fondo por modelo externo:
    # evitar imports innecesarios (pyrender/EGL, cv2 y RMBG) que pueden fallar en Colab.
    optional_imports=[
        "from src.utils.render_utils import render_views_around_mesh, render_normal_views_around_mesh, make_grid_for_images_or_videos, export_renderings",
        "from src.utils.image_utils import prepare_image",
        "from src.models.briarmbg import BriaRMBG"
    ]
    for imp in optional_imports:
        marker="# HIGHFLY_R06_OPTIONAL_IMPORT_DISABLED: "+imp
        if marker in text:
            continue  # reejecución: ya fue desactivado
        if imp in text:
            text=text.replace(imp,marker)
        else:
            raise RuntimeError("Import opcional de PartCrafter cambió: "+imp)
    old_a='    snapshot_download(repo_id="briaai/RMBG-1.4", local_dir=rmbg_weights_dir)'
    old_b='    rmbg_net = BriaRMBG.from_pretrained(rmbg_weights_dir).to(device)\n    rmbg_net.eval()'
    old_patched_a='    if args.rmbg:\n        snapshot_download(repo_id="briaai/RMBG-1.4", local_dir=rmbg_weights_dir)'
    old_patched_b='    rmbg_net = None\n    if args.rmbg:\n        rmbg_net = BriaRMBG.from_pretrained(rmbg_weights_dir).to(device)\n        rmbg_net.eval()'
    for snippet in (old_patched_a,old_a):
        if snippet in text:
            text=text.replace(snippet,"    # HIGHFLY_R06_RMBG_MODEL_DISABLED")
    for snippet in (old_patched_b,old_b):
        if snippet in text:
            text=text.replace(snippet,"    rmbg_net = None  # HIGHFLY_R06_RMBG_DISABLED")
    if "HIGHFLY_R06_RMBG_MODEL_DISABLED" not in text or "HIGHFLY_R06_RMBG_DISABLED" not in text:
        raise RuntimeError("No se pudo deshabilitar RMBG sin tocar código desconocido")
    if "    if args.render:\n" in text:
        text=text.replace("    if args.render:\n", "    if False and args.render:  # HIGHFLY_R06_NO_RENDER\n")
    elif "HIGHFLY_R06_NO_RENDER" not in text:
        raise RuntimeError("PartCrafter cambió su bloque de render")
    # Prohibir explícitamente el uso del modelo no comercial en nuestro lanzamiento.
    anchor='    args = parser.parse_args()'
    assert anchor in text
    guard='\n    if args.rmbg or args.render or args.part_suggest or args.style_transfer:\n        raise RuntimeError("HIGHFLY gratuito: render, RMBG y servicios VLM deshabilitados")'
    if guard not in text:
        text=text.replace(anchor,anchor+guard)
    src.write_text(text,encoding="utf-8")
    print("Modo exclusivo geometría 3D: sin render, sin RMBG, sin proveedores con créditos.",flush=True)

try:
    stage("1/6 · COMPROBAR GPU GRATUITA")
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Colab no tiene GPU CUDA asignada. Seleccioná GPU T4 gratuita en Entorno de ejecución, sin pagar.")
    props=torch.cuda.get_device_properties(0)
    gb=props.total_memory/1024**3
    print("GPU:",props.name,"| VRAM:",round(gb,2),"GiB",flush=True)
    if gb<7.5:
        raise RuntimeError("PartCrafter requiere al menos 8 GiB VRAM. No se realizará inferencia.")
    # No activar automáticamente acceso a servicios externos por IA.
    os.environ.pop("GEMINI_API_KEY",None)

    stage("2/6 · ELEGIR IMAGEN OBSIDIAN MONARCH")
    from google.colab import files
    from PIL import Image, ImageOps
    uploaded=files.upload()
    if len(uploaded)!=1:
        raise RuntimeError("Subí una sola imagen de referencia PNG/JPG/WEBP.")
    name, raw=next(iter(uploaded.items()))
    if not name.lower().endswith((".png",".jpg",".jpeg",".webp")):
        raise RuntimeError("Formato inválido; subir PNG/JPG/WEBP.")
    if len(raw)>20*1024*1024:
        raise RuntimeError("La imagen excede el máximo de 20 MiB.")
    with Image.open(BytesIO(raw)) as img:
        img=ImageOps.exif_transpose(img).convert("RGBA")
        canvas_rgba=Image.new("RGBA",img.size,(255,255,255,255))
        canvas_rgba.alpha_composite(img)
        rgb=canvas_rgba.convert("RGB")
        rgb.thumbnail((768,768),Image.Resampling.LANCZOS)
    background=Image.new("RGB",(768,768),"white")
    background.paste(rgb,((768-rgb.width)//2,(768-rgb.height)//2))
    prepared=Path("/content/highfly_obsidian_reference.png")
    background.save(prepared)
    display(background.resize((320,320)))
    print("Imagen preparada:",prepared,flush=True)

    stage("3/6 · INSTALACIÓN REUTILIZABLE PYTHON 3.11")
    if not ROOT.is_dir():
        run(["git","clone","--no-checkout","https://github.com/wgsxm/PartCrafter.git",ROOT],timeout=700)
    run(["git","checkout",CHECKPOINT],cwd=ROOT,timeout=120)
    run(["git","-c","advice.detachedHead=false","rev-parse","HEAD"],cwd=ROOT)
    if not shutil.which("uv"):
        run([sys.executable,"-m","pip","install","-q","uv"],timeout=300)
    uv=shutil.which("uv") or str(Path(sys.executable).parent/"uv")
    run([uv,"python","install","3.11"],timeout=600)
    if not PY.exists():
        run([uv,"venv","--seed","--python","3.11",VENV],timeout=300)
    run([PY,"-c","import sys; assert sys.version_info[:2]==(3,11),sys.version"])
    # Si venimos de la ejecución fallida de apt, se reaprovechan ruedas descargadas.
    marker=VENV/".highfly_r05_installed"
    if not marker.exists():
        run([PY,"-m","pip","install","-q",
             "torch==2.5.1","torchvision==0.20.1","torchaudio==2.5.1",
             "--index-url","https://download.pytorch.org/whl/cu124"],timeout=1600)
        run([PY,"-m","pip","install","-q","--only-binary=:all:",
             "torch-cluster","-f","https://data.pyg.org/whl/torch-2.5.1+cu124.html"],timeout=800)
        run([PY,"-m","pip","install","-q","-r","settings/requirements.txt"],cwd=ROOT,timeout=1800)
        marker.write_text("Installed in free Colab Python 3.11; real inference not guaranteed.",encoding="utf-8")
    else:
        print("Dependencias existentes reutilizadas (sin reinstalar).",flush=True)

    # El requirements.txt oficial NO fija versiones. Versiones futuras de
    # diffusers/transformers pueden eliminar clases que PartCrafter importa.
    # Este paso se hace UNA sola vez por entorno; reutiliza torch+cu124 anterior.
    pin_marker=VENV/".highfly_r06_libraries_pinned"
    if not pin_marker.exists():
        stage("3/6 · FIJAR LIBRERÍAS COMPATIBLES (SIN CRÉDITOS)")
        run([PY,"-m","pip","install","-q",
             "diffusers==0.34.0","transformers==4.52.4",
             "huggingface-hub==0.34.4","accelerate==1.7.0",
             "peft==0.16.0"],cwd=ROOT,timeout=1000)
        pin_marker.write_text("diffusers0.34 transformers4.52.4 HF0.34.4",encoding="utf-8")
    # No apt-get: el antiguo paquete libgl1-mesa-dev ya falló con error 100.
    run([PY,"-c","import sys,torch,torch_cluster,trimesh; assert torch.cuda.is_available(); print('HIGHFLY CUDA PRECHECK OK',torch.__version__,torch.cuda.get_device_name(0))"],cwd=ROOT,timeout=120)
    verify_partcrafter_patches()
    run([PY,"-c",
         "from src.pipelines.pipeline_partcrafter import PartCrafterPipeline; from transformers import BitImageProcessor, Dinov2Model; import diffusers,transformers; print('HIGHFLY PARTCRAFTER IMPORT OK', diffusers.__version__, transformers.__version__)"],
         cwd=ROOT,timeout=180)

    stage("4/6 · GENERAR 4 PIEZAS 3D CON PARTCRAFTER")
    # No invocar Gemini, VLM pagos, remover fondo con RMBG ni render opcional.
    cmd=[PY,"scripts/inference_partcrafter.py",
         "--image_path",prepared,"--num_parts","4",
         "--num_tokens","512","--num_inference_steps","25",
         "--output_dir",OUTDIR,"--tag",TAG,"--seed","17"]
    assert all(flag not in cmd for flag in ("--rmbg","--part_suggest","--style_transfer","--render"))
    env=dict(os.environ,PYOPENGL_PLATFORM="egl",GEMINI_API_KEY="")
    run(cmd,cwd=ROOT,timeout=3000,env=env)

    stage("5/6 · VALIDAR ARCHIVOS GLB Y CREAR ZIP")
    expected=[TARGET/"object.glb",TARGET/"manifest.json"]+[TARGET/f"part_{i:02}.glb" for i in range(4)]
    for f in expected:
        if not f.is_file() or f.stat().st_size<100:
            raise RuntimeError("No se generó un archivo real y utilizable: "+str(f))
    manifest=json.loads((TARGET/"manifest.json").read_text())
    if manifest.get("num_parts")!=4:
        raise RuntimeError("El manifiesto no declara 4 piezas.")
    # Inspección real de vértices/caras desde el Python aislado.
    check_script=r"""
import sys, trimesh, json, math
from pathlib import Path
p=Path(sys.argv[1])
summary={}
for file in [p/'object.glb',*sorted(p.glob('part_*.glb'))]:
    scene=trimesh.load(file,force='scene')
    meshes=[m for m in scene.geometry.values() if isinstance(m,trimesh.Trimesh)]
    vertices=sum(len(m.vertices) for m in meshes)
    faces=sum(len(m.faces) for m in meshes)
    if vertices<20 or faces<15 or any(not math.isfinite(float(x)) for m in meshes for x in m.bounds.flatten()):
        raise RuntimeError('GEOMETRIA_VACIA_O_DUMMY: '+file.name)
    summary[file.name]={'vertices':vertices,'faces':faces}
print('HIGHFLY REAL GLB MESH CHECK',json.dumps(summary))
"""
    run([PY,"-c",check_script,TARGET],cwd=ROOT,timeout=240)
    sha={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in expected}
    provenance={
        "schema":"highfly.partcrafter.raw-3d/v1",
        "provider":"PartCrafter image-to-multipart, no credit API",
        "repository":"wgsxm/PartCrafter","commit":CHECKPOINT,
        "gpu":props.name,"input":"user uploaded image / white background / no RMBG or VLM",
        "generatedParts":4,"riggedToClaudeCraft":False,"artApproved":False,
        "fileSha256":sha,"createdUtc":datetime.now(timezone.utc).isoformat()
    }
    metadata=TARGET/"highfly-provenance.json"
    metadata.write_text(json.dumps(provenance,indent=2,ensure_ascii=False),encoding="utf-8")
    with zipfile.ZipFile(ZIP,"w",zipfile.ZIP_DEFLATED) as archive:
        for f in [*expected,metadata]:
            archive.write(f,arcname=f.name)
    print("ZIP GENERADO:",ZIP,"| MiB:",round(ZIP.stat().st_size/1024**2,2),flush=True)

    stage("6/6 · DESCARGA AUTOMÁTICA DEL ZIP")
    print("HIGHFLY ✅ GEOMETRÍA 3D GENERADA. FALTA FITTING + RIG EN BLENDER.",flush=True)
    files.download(str(ZIP))

except Exception as err:
    print("\n"+"!"*64)
    print("HIGHFLY ERROR REAL EN ETAPA:",CURRENT_STAGE)
    print("TIPO:",type(err).__name__)
    print("DETALLE:",str(err))
    print("No se ha generado ni aprobado una armadura jugable.")
    print("Copiá esta parte del error si necesitás que lo corrija en GitHub.")
    print("!"*64,flush=True)
    raise
